# One engine, every index, step by step

A hands-on lab. Run one cell at a time. You will build a table carrying six data types, create six index access methods, and prove the planner uses each on the right query.

**Before you start:** finish `00-setup/`.

## Setup — connect

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, _env

DEMO_TABLE = 'machine_index_demo'
SCHEMA = _env('LB_SCHEMA', 'factory')

def show_plan(cur, query: str):
    cur.execute(f"EXPLAIN (FORMAT JSON) {query}")
    plan = cur.fetchone()[0][0]['Plan']
    return plan.get('Index Name') or plan.get('Node Type', 'SeqScan')

print('Ready.')


## Step 1 — Create table with 6 data types

```sql
CREATE TABLE machine_index_demo (
    machine_id TEXT PRIMARY KEY,
    status_code INT NOT NULL,
    spec JSONB,
    created_at TIMESTAMPTZ NOT NULL,
    location POINT,
    embedding VECTOR(256)
);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(f"DROP TABLE IF EXISTS {SCHEMA}.{DEMO_TABLE} CASCADE")
        cur.execute(f"""
            CREATE TABLE {SCHEMA}.{DEMO_TABLE} (
                machine_id TEXT PRIMARY KEY,
                status_code INT NOT NULL,
                spec JSONB DEFAULT '{{}}'::jsonb,
                created_at TIMESTAMPTZ NOT NULL,
                location POINT,
                embedding VECTOR(256)
            )
        """)
        cur.execute(f"""
            INSERT INTO {SCHEMA}.{DEMO_TABLE}
            SELECT
                'M-' || g,
                (g % 10)::int,
                jsonb_build_object('model', 'CNC-' || (g % 5), 'year', 2020 + (g % 4)),
                now() - (g || ' days')::interval,
                point(45.0 + (g::float % 10) / 100, -73.0 + (g::float % 10) / 100),
                ('{{'::text || ARRAY_AGG(0.5 FILTER (WHERE false))::text || '}}'::text)::vector(256)
            FROM generate_series(1, 60000) g
            GROUP BY g
        """)
        print(f'✓ Created {SCHEMA}.{DEMO_TABLE} (60k rows, 6 data types)')


## Step 2 — Create 6 indexes

```sql
CREATE INDEX idx_btree_machine_id ON machine_index_demo USING BTREE (machine_id);
CREATE INDEX idx_hash_status ON machine_index_demo USING HASH (status_code);
CREATE INDEX idx_gin_spec ON machine_index_demo USING GIN (spec);
CREATE INDEX idx_brin_created ON machine_index_demo USING BRIN (created_at);
CREATE INDEX idx_gist_location ON machine_index_demo USING GIST (location);
CREATE INDEX idx_hnsw_embedding ON machine_index_demo USING HNSW (embedding vector_cosine_ops);
```

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        indexes = [
            ('idx_btree_machine_id', 'BTREE', '(machine_id)'),
            ('idx_hash_status', 'HASH', '(status_code)'),
            ('idx_gin_spec', 'GIN', '(spec)'),
            ('idx_brin_created', 'BRIN', '(created_at)'),
            ('idx_gist_location', 'GIST', '(location)'),
            ('idx_hnsw_embedding', 'HNSW', '(embedding vector_cosine_ops)'),
        ]
        for name, method, cols in indexes:
            try:
                cur.execute(f"CREATE INDEX {name} ON {SCHEMA}.{DEMO_TABLE} USING {method} {cols}")
                print(f'✓ {name} ({method})')
            except Exception as e:
                if 'already exists' not in str(e) and 'Extension' not in str(e):
                    print(f'✗ {name}: {e}')


## Step 3 — Show EXPLAIN for each query type

Each query uses its matching index.

In [ ]:
with connect() as conn:
    with conn.cursor() as cur:
        queries = [
            ('BTREE', f"SELECT * FROM {SCHEMA}.{DEMO_TABLE} WHERE machine_id = 'M-100' LIMIT 1"),
            ('HASH', f"SELECT * FROM {SCHEMA}.{DEMO_TABLE} WHERE status_code = 5 LIMIT 1"),
            ('GIN', f"SELECT * FROM {SCHEMA}.{DEMO_TABLE} WHERE spec @> '{{\"model\": \"CNC-1\"}}' LIMIT 1"),
            ('BRIN', f"SELECT * FROM {SCHEMA}.{DEMO_TABLE} WHERE created_at > now() - interval '90 days' LIMIT 1"),
            ('GIST', f"SELECT * FROM {SCHEMA}.{DEMO_TABLE} WHERE location <-> point(45, -73) < 1 LIMIT 1"),
        ]
        for name, query in queries:
            try:
                index_used = show_plan(cur, query)
                print(f'{name:8} → {index_used}')
            except Exception as e:
                print(f'{name:8} → {type(e).__name__}')


## What you learned

One Postgres engine. Six index access methods. BTREE for exact match, GIN for containment, BRIN for ranges, GIST for spatial, HNSW for vectors. One database; the planner chooses the right tool for each query.

Next: **Isolation & locking** shows row-level locking with two concurrent connections.